In [1]:
import os
import numpy as np

from utils import extract_features

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import classification_report

In [2]:
def load_data(split="training", limit=200):
    X = []
    y = []

    base_path = r"C:\Users\NIHARIKA\OneDrive\Documents\Professional Work\Projects\deepfake-audio-detector\raw_dataset"
    sections = ["for-norm", "for-2sec", "for-original", "for-rerec"]

    def find_path(section_path):
        for root, dirs, files in os.walk(section_path):
            if "real" in dirs and "fake" in dirs and split in root:
                return root
        return None

    for section in sections:
        print(f"\nProcessing {split}: {section}")

        section_path = os.path.join(base_path, section)
        split_path = find_path(section_path)

        if split_path is None:
            print("Not found")
            continue

        print("Found:", split_path)

        for label in ["real", "fake"]:
            path = os.path.join(split_path, label)

            files = os.listdir(path)[:limit]

            for file in files:
                file_path = os.path.join(path, file)

                features = extract_features(file_path)

                if features is not None:
                    X.append(features)
                    y.append(0 if label == "real" else 1)

    X = np.array(X)[..., np.newaxis]
    y = np.array(y)

    return X, y

In [3]:
X_train, y_train = load_data("training", limit=500)
X_val, y_val = load_data("validation", limit=200)
X_test, y_test = load_data("testing", limit=200)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


Processing training: for-norm
Found: C:\Users\NIHARIKA\OneDrive\Documents\Professional Work\Projects\deepfake-audio-detector\raw_dataset\for-norm\for-norm\training

Processing training: for-2sec
Found: C:\Users\NIHARIKA\OneDrive\Documents\Professional Work\Projects\deepfake-audio-detector\raw_dataset\for-2sec\for-2seconds\training

Processing training: for-original
Found: C:\Users\NIHARIKA\OneDrive\Documents\Professional Work\Projects\deepfake-audio-detector\raw_dataset\for-original\for-original\training

Processing training: for-rerec
Found: C:\Users\NIHARIKA\OneDrive\Documents\Professional Work\Projects\deepfake-audio-detector\raw_dataset\for-rerec\for-rerecorded\training

Processing validation: for-norm
Found: C:\Users\NIHARIKA\OneDrive\Documents\Professional Work\Projects\deepfake-audio-detector\raw_dataset\for-norm\for-norm\validation

Processing validation: for-2sec
Found: C:\Users\NIHARIKA\OneDrive\Documents\Professional Work\Projects\deepfake-audio-detector\raw_dataset\for-2se

In [4]:
model = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=(128,128,1)),
    MaxPooling2D(2,2),

    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

C:\Users\NIHARIKA\OneDrive\Documents\Projects\deepfake_audio_detection\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss=BinaryCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

In [6]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

Epoch 1/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 9s 59ms/step - accuracy: 0.6867 - loss: 0.5851 - val_accuracy: 0.7406 - val_loss: 0.5432
Epoch 2/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 54ms/step - accuracy: 0.7805 - loss: 0.5068 - val_accuracy: 0.8213 - val_loss: 0.4722
Epoch 3/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 54ms/step - accuracy: 0.8338 - loss: 0.4539 - val_accuracy: 0.8631 - val_loss: 0.4311
Epoch 4/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.8780 - loss: 0.4078 - val_accuracy: 0.8694 - val_loss: 0.4133
Epoch 5/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.8905 - loss: 0.3844 - val_accuracy: 0.8931 - val_loss: 0.3820
Epoch 6/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - accuracy: 0.9110 - loss: 0.3577 - val_accuracy: 0.9131 - val_loss: 0.3551
Epoch 7/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.9170 - loss: 0.3488 - val_accuracy: 0.8988 - val_loss: 0.3691
Epoch 8/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 54ms/step - accuracy: 0.9320 - loss: 0.3316 - val_accu

In [7]:
y_pred = (model.predict(X_test) > 0.5).astype(int)

print(classification_report(y_test, y_pred))

50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step
              precision    recall  f1-score   support

           0       0.81      0.92      0.86       800
           1       0.91      0.79      0.85       800

    accuracy                           0.86      1600
   macro avg       0.86      0.86      0.85      1600
weighted avg       0.86      0.86      0.85      1600



In [8]:
pred = model.predict(X_test[:10]).flatten()

print("Predictions:", pred)
print("Actual:", y_test[:10])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
Predictions: [0.030227   0.04610613 0.39559054 0.21852078 0.07412599 0.52577776
 0.01674776 0.4750934  0.02464021 0.05453759]
Actual: [0 0 0 0 0 0 0 0 0 0]


In [9]:
model.save("deepfake_detector.h5")
print("Model saved!")

Model saved!
